# Lecture 2.3 — Reading Agent Messages & the Result Stream

**Course:** Build Production AI Agents with the Claude Agent SDK
**Section 2:** Your First Agent (Quickstart)

---

In Lecture 2.2 you wrote your first `query()` call and saw raw output stream past.
This notebook decodes that output completely.

**The foundational insight for this lecture:**
> The SDK is the only thing that writes to your stream — always.
> Claude returns JSON over HTTP. The SDK constructs every Python object you see.
> The distinction between messages is purely about who *decided* the content — not who surfaced it.

**What we cover:**
1. Revisit the raw output from 2.2 — name every object
2. Walk through message types and content blocks with labelled code
3. The clean production pattern — `hasattr(message, "result")`
4. When you would read content blocks in real-world work

---
## Install & API Key Setup

In [1]:
# ── Setup (carried over from Lecture 2.2) ──────────────────────────────────

# We pin the Claude Agent SDK to a specific version to ensure all examples
# in this notebook run exactly as shown in the course. If you encounter any
# issues or want to experiment with newer features, you can install the latest
# version by removing the version pin (replace 'claude-agent-sdk==0.2.93' with just
# 'claude-agent-sdk'). Note that newer versions may behave differently from
# what is demonstrated in the videos. We will update the notebooks periodically
# to keep up with new releases.

!pip install claude-agent-sdk==0.2.93 -q

import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
print("✅ SDK installed and API key set.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 MB 11.5 MB/s eta 0:00:00
✅ SDK installed and API key set.


---
## Raw Output Revisited

---
## Part 1 — Revisiting the Raw Output from Lecture 2.2

Let's re-run the exact same `query()` call from 2.2 — same prompt, same tools.
This time, read every line of output carefully before moving on.

In [2]:
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions


async def main():
    async for message in query(
        prompt="What files are in this directory?",
        options=ClaudeAgentOptions(allowed_tools=["Bash", "Glob"], model="claude-haiku-4-5",),
    ):
        print(message)


await main()

SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/content', 'session_id': '5fc77a6d-b439-4da8-8aca-edbc05d9d915', 'tools': ['Task', 'AskUserQuestion', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'Edit', 'EnterPlanMode', 'EnterWorktree', 'ExitPlanMode', 'ExitWorktree', 'Glob', 'Grep', 'Monitor', 'NotebookEdit', 'PushNotification', 'Read', 'ScheduleWakeup', 'Skill', 'TaskCreate', 'TaskGet', 'TaskList', 'TaskOutput', 'TaskStop', 'TaskUpdate', 'ToolSearch', 'WebFetch', 'WebSearch', 'Workflow', 'Write'], 'mcp_servers': [], 'model': 'claude-haiku-4-5', 'permissionMode': 'default', 'slash_commands': ['deep-research', 'update-config', 'verify', 'debug', 'code-review', 'simplify', 'batch', 'fewer-permission-prompts', 'loop', 'claude-api', 'run', 'run-skill-generator', 'clear', 'compact', 'context', 'heapdump', 'init', 'reload-skills', 'review', 'security-review', 'usage', 'insights', 'goal', 'team-onboarding'], 'apiKeySource': 'ANTHROPIC_API_KEY', 'claude_cod

---
## Message Types & Content Blocks

---
## Part 2 — The Message Types & Content Blocks

### The four top-level message types

| Message Type | Generated by | When | Key fields |
|---|---|---|---|
| `SystemMessage` | **SDK** | Before Claude sees anything | `subtype`, `data` (has `session_id`) |
| `AssistantMessage` | **Claude** (packaged by SDK) | Every Claude response | `content` (list of blocks) |
| `UserMessage` | **SDK** | After each tool executes | `content` (list of blocks) |
| `ResultMessage` | **SDK** | Once, loop complete | `result`, `total_cost_usd`, `num_turns`, `duration_ms` |

> ⚠️ **No `ToolUseMessage` or `ToolResultMessage`** — those types do not exist in the Python SDK.
> Tool activity lives inside `AssistantMessage` and `UserMessage` as **content blocks**.

---

### Content blocks inside the messages

**Inside `AssistantMessage.content`:**
| Block | Key fields | What it means |
|---|---|---|
| `TextBlock` | `block.text` | Claude's plain text output |
| `ToolUseBlock` | `block.name`, `block.input` | Claude calling a tool |
| `ThinkingBlock` | `block.thinking` | Claude's extended reasoning (thinking mode) |

**Inside `UserMessage.content`:**
| Block | Key fields | What it means |
|---|---|---|
| `ToolResultBlock` | `block.content` | Result returned from a tool execution |

---

### The two-level pattern
```
Check isinstance on the outer message
  └── if AssistantMessage or UserMessage:
        loop through message.content
        └── check isinstance on each block
```

---

### Your prompt is never in the stream
The SDK packages your `prompt=` string and sends it to Claude internally.
It never appears in your `async for` loop. The first thing you always see is `SystemMessage` from the SDK.

In [3]:
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    SystemMessage,
    AssistantMessage,
    UserMessage,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    ToolResultBlock,
    ThinkingBlock,
)


async def main():
    async for message in query(
        prompt="What files are in this directory?",
        options=ClaudeAgentOptions(allowed_tools=["Bash", "Glob"], model="claude-haiku-4-5",),
    ):
        if isinstance(message, SystemMessage):
            # Generated by the SDK — before Claude sees anything
            print(f"[SYSTEM]       SDK init — subtype={message.subtype}")

        elif isinstance(message, AssistantMessage):
            # Content decided by Claude, packaged by SDK
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(f"[TEXT]         {block.text[:80]}")
                elif isinstance(block, ToolUseBlock):
                    print(f"[TOOL USE]     tool={block.name}  input={block.input}")
                elif isinstance(block, ThinkingBlock):
                    print(f"[THINKING]     {block.thinking[:60]}...")

        elif isinstance(message, UserMessage):
            # Generated by the SDK — tool result fed back to Claude
            for block in message.content:
                if isinstance(block, ToolResultBlock):
                    print(f"[TOOL RESULT]  {str(block.content)[:80]}")

        elif isinstance(message, ResultMessage):
            # Generated by the SDK — loop complete, once only
            print(f"[RESULT]       {message.result[:80]}")
            print(f"               cost=${message.total_cost_usd:.4f}  "
                  f"turns={message.num_turns}  "
                  f"duration={message.duration_ms}ms")


await main()

# ──────────────────────────────────────────────────────────────────────────────
# Pattern: two levels
#   1. isinstance on the outer message
#   2. loop through message.content → isinstance on each block
#
# [:80] is just to keep notebook output tidy — in production read full content
# ──────────────────────────────────────────────────────────────────────────────

[SYSTEM]       SDK init — subtype=init
[SYSTEM]       SDK init — subtype=thinking_tokens
[SYSTEM]       SDK init — subtype=thinking_tokens
[SYSTEM]       SDK init — subtype=thinking_tokens
[SYSTEM]       SDK init — subtype=thinking_tokens
[SYSTEM]       SDK init — subtype=thinking_tokens
[THINKING]     The user is asking what files are in "this directory". They ...
[TOOL USE]     tool=Bash  input={'command': 'ls -la', 'description': 'List files in current directory'}
[TOOL RESULT]  total 16
drwxr-xr-x 1 root root 4096 Jun  4 13:32 .
drwxr-xr-x 1 root root 4096 
[TEXT]         The current directory contains:

1. **`.config/`** — A configuration directory (
[RESULT]       The current directory contains:

1. **`.config/`** — A configuration directory (
               cost=$0.0092  turns=2  duration=2997ms


---
## The Clean Production Pattern

---
## Part 3 — The Clean Production Pattern

In most real agents — a CI/CD pipeline, a backend service, a user-facing app —
you only need the **final answer**. Here is the pattern you will use throughout this course.

### Why `hasattr(message, "result")` and not `TextBlock`?

`AssistantMessage` with `TextBlock` can appear **multiple times** during a run —
including as intermediate commentary between tool calls.

`ResultMessage` appears **exactly once**, at the very end.
Its `.result` field contains the **same text** as the last `AssistantMessage` `TextBlock`
— the SDK copies it there — but it also gives you:
- `total_cost_usd` — cost of the full run
- `num_turns` — how many tool call rounds happened
- `duration_ms` — total elapsed time
- `is_error` — whether the run ended in an error

`hasattr(message, "result")` waits for the SDK's signal that the task is **fully complete**.

In [4]:
async def main():
    async for message in query(
        prompt="What files are in this directory?",
        options=ClaudeAgentOptions(allowed_tools=["Bash", "Glob"], model="claude-haiku-4-5",),
    ):
        if hasattr(message, "result"):
            print(message.result)


await main()

# ──────────────────────────────────────────────────────────────────────────────
# WHY hasattr?
#   Only ResultMessage has a .result field.
#   This fires exactly once — when the SDK signals the loop is fully complete.
#   The text is identical to the last AssistantMessage TextBlock.
#   But ResultMessage also carries cost/turn/duration metadata.
#
# ALTERNATIVE (equally correct, more explicit):
#   if isinstance(message, ResultMessage):
#       print(message.result)
#
# DEFAULT throughout this course: hasattr(message, "result")
# ──────────────────────────────────────────────────────────────────────────────

The current directory contains:

1. **.config/** — A configuration directory
2. **sample_data/** — A directory containing sample data

Both are directories (indicated by the `d` at the start of their permissions). There are no regular files in the current directory—just these two subdirectories.

Would you like me to explore what's inside either of these directories?


---
## When Would You Read Content Blocks?

---
## Part 4 — When Would You Read Content Blocks?

`hasattr(message, "result")` is the right default. But here are three real-world
scenarios where reading content blocks is the correct approach:

---

### 🔍 Debugging

Loop through `AssistantMessage.content` and read `ToolUseBlock` to see exactly what
tool was called and what input was passed. Read `ToolResultBlock` from `UserMessage`
to see what came back.

```python
elif isinstance(message, AssistantMessage):
    for block in message.content:
        if isinstance(block, ToolUseBlock):
            print(f"Tool called: {block.name}")
            print(f"Input:       {block.input}")

elif isinstance(message, UserMessage):
    for block in message.content:
        if isinstance(block, ToolResultBlock):
            print(f"Result:      {block.content}")
```

---

### 📺 Real-Time UI Progress

As `ToolUseBlock` objects arrive inside `AssistantMessage`, update a progress
indicator in your frontend — *"Claude is searching..."*, *"Running tests..."*

---

### 📋 Audit Logging (→ Section 5: Hooks)

Log every `ToolUseBlock` to a file for compliance or auditing.

---

**Summary:** Use `hasattr(message, "result")` by default.
Read content blocks when you need visibility, real-time feedback, or audit trails.

---
## Summary — Lecture 2.3

### What we established

**The foundational insight:**
The SDK is the only thing that writes to your stream. Claude returns JSON over HTTP.
The SDK constructs every Python object. Who decides ≠ who packages.

**Four message types:**
- `SystemMessage` — SDK, before Claude sees anything
- `AssistantMessage` — Claude decides, SDK packages; carries content blocks
- `UserMessage` — SDK, carries `ToolResultBlock`; tool results come back here
- `ResultMessage` — SDK, once, loop complete; same text as last `TextBlock` + metadata

**No `ToolUseMessage` or `ToolResultMessage`** — tool activity is always a block inside a message.

**Two-level pattern:** `isinstance` on message → loop `message.content` → `isinstance` on block.

**Production pattern:** `hasattr(message, "result")` — fires once, on `ResultMessage`, task done.

**Your prompt is never echoed back** — first thing in stream is always `SystemMessage` from the SDK.

### Key decision
`hasattr(message, "result")` is the default pattern throughout this course.
Use `isinstance(message, ResultMessage)` if you prefer explicit typing.

---
**Coming up in Lecture 2.4 →**
*Hands-On: Build a File Explorer Agent* — everything from Section 2 in one complete agent.